# Python you need to read
*Python for Healthcare · Stanford Technology Training (ITS-2519) · Dr. Lionel Levine*
**Session 1 · about 40 minutes**

An AI assistant can write Python for you in seconds. You still have to read it, because you're the one who
vouches for the result. This notebook covers the Python you need to *read* an analysis, not to write one from memory.

**Work through it the same way every time: predict, then run.**
1. Read the code cell. Before you run it, decide what you expect it to show.
2. Run it with **Shift + Enter**.
3. If the output surprised you, that's the useful part. Work out why before you move on.

**Gemini's Learn Mode.** Open the Gemini panel and switch on **Learn Mode**. In Learn Mode, Gemini coaches and
explains instead of writing code for you. Try it on any cell: *Explain this cell line by line.*
(Can't find Learn Mode? Start your message with "Don't write code for me.")

## 1. Values and names
A name (a *variable*) stores a value so you can use it later. In Python, `=` means *store*, not *equals*.

**Predict:** what will the second `print` line show?

In [ ]:
sbp = 152
dbp = 94
participant = "KV1003"

print(participant, sbp, dbp)
print(sbp >= 140)

Comparisons such as `>=`, `<`, and `==` answer with `True` or `False`. You'll see them inside almost every filter.

## 2. A calculation
**Predict:** roughly what number will this show? Multiplication happens before addition, as on paper.

In [ ]:
mean_arterial_pressure = (sbp + 2 * dbp) / 3
round(mean_arterial_pressure, 1)

The last line of a cell displays its value even without `print`.

## 3. Lists: several values in order
**Predict:** what will each of the four lines print?

In [ ]:
readings = [152, 148, 141, 139]      # SBP at weeks 0, 4, 8, and 12

print(len(readings))
print(readings[0])
print(readings[-1])
print(readings[-1] - readings[0])

Python counts from 0, so `readings[0]` is the first reading (week 0), and `[-1]` means the last one.
Those two facts explain many off-by-one surprises in AI-written code.

## 4. Dictionaries: values looked up by name

In [ ]:
person = {"id": "KV1003", "site": "North", "age": 39, "arm": "coaching"}

print(person["age"])
person["age"] = 40
print(person)

A dictionary finds a value by its name (its *key*). A row of a table works much the same way.

## 5. Functions: a reusable recipe
**Predict:** what will the two lines print?

In [ ]:
def map_mmhg(sbp, dbp):
    """Estimated mean arterial pressure, in mmHg."""
    return (sbp + 2 * dbp) / 3

print(round(map_mmhg(152, 94), 1))
print(round(map_mmhg(120, 80), 1))

`def` defines the recipe. The indented lines belong to it, and `return` hands back the result.
In Python, indentation is part of the grammar, not decoration.

## 6. A table: the pandas DataFrame
`pandas` is Python's table library. By convention it's imported as `pd`, and a table is a *DataFrame*.
Nearly every analysis an assistant writes for you starts here.

In [ ]:
import pandas as pd

clinic = pd.DataFrame({
    "id":   ["A01", "A02", "A03", "A04", "A05", "A06", "A07", "A08"],
    "site": ["North", "North", "South", "South", "North", "South", "North", "South"],
    "arm":  ["coaching", "usual care", "coaching", "usual care",
             "coaching", "usual care", "usual care", "coaching"],
    "sbp":  [152, 147, 160, 143, 155, 149, 158, 141],
    "dbp":  [94, 90, 99, 88, 92, 91, 97, 86],
})
clinic

**Predict:** what will `shape` show?

In [ ]:
print(clinic.shape)          # (rows, columns)
print(list(clinic.columns))
clinic.describe()

Check `shape` often. It's the quickest way to notice rows that appeared or vanished.

## 7. Whole-column math
One line computes a new column for every row at once, with no loop.

In [ ]:
clinic["map"] = (clinic["sbp"] + 2 * clinic["dbp"]) / 3
clinic.round(1)

Underneath, each column is a NumPy array, which is what makes whole-column math fast:

In [ ]:
clinic["sbp"].to_numpy()

## 8. Pick columns, keep rows, sort
Double brackets pick a list of columns:

In [ ]:
clinic[["id", "sbp"]]

**Predict:** how many rows will be left?

In [ ]:
clinic[clinic["sbp"] >= 150]

`.loc[rows, columns]` keeps rows by a condition and picks columns by name, in one step. `==` asks *is it equal?*

In [ ]:
clinic.loc[clinic["site"] == "South", ["id", "arm", "sbp"]]

In [ ]:
clinic.sort_values("sbp", ascending=False).head(3)

## 9. Count categories, and a trap

In [ ]:
clinic["arm"].value_counts()

**Predict:** how many different values will the first count show? And the second?

In [ ]:
labels = pd.Series(["coaching", "coaching ", "Coaching", "usual care"])
print(labels.value_counts())
print()
print(labels.str.strip().str.lower().value_counts())

Two lines in the first count look identical: one of them ends with a space you can't see. To Python,
`"coaching "` (with a trailing space) and `"Coaching"` are both different values from `"coaching"`.
`.str.strip()` removes the extra spaces and `.str.lower()` makes the text lowercase. You'll meet this in Lab 1.

## 10. Group and summarize
**Predict:** which arm has the higher mean SBP?

In [ ]:
clinic.groupby("arm")["sbp"].mean()

In [ ]:
clinic.groupby("arm")["sbp"].agg(["count", "mean", "std"]).round(1)

Read `groupby` as: *split the rows by arm, then summarize each group's SBP.* Always look at `count` too.
A mean of 2 people and a mean of 200 look the same.

## 11. Joining two tables
`labs` has one row per lab result, so some patients have several rows.

**Predict:** how many rows will `merged` have? (A01 has 2 lab results, A02 has 1, A03 has 3, and the others have none.)

In [ ]:
labs = pd.DataFrame({
    "id":    ["A01", "A01", "A02", "A03", "A03", "A03"],
    "test":  ["K", "Cr", "K", "K", "Cr", "A1c"],
    "value": [4.1, 0.9, 3.8, 4.5, 1.1, 6.2],
})

merged = clinic.merge(labs, on="id", how="left")
print(len(clinic), "rows before the merge")
print(len(merged), "rows after the merge")

A merge repeats each patient's row once for every matching lab row. Patients with no labs keep one row, with blanks
(`NaN`). If you wanted one row per patient, that growth is a bug, and an easy one to miss.

pandas can refuse a merge that would repeat rows. `validate="one_to_one"` is a **stop rule**: code that refuses
to continue when the data break an assumption. You'll write your own in Lab 1.

In [ ]:
try:
    clinic.merge(labs, on="id", how="left", validate="one_to_one")
except Exception as err:
    print("pandas refused:", err)

## 12. Reading an error message
The next cell fails on purpose. Run it, then read the **last line** of the error first.

In [ ]:
clinic["SBP"].mean()

`KeyError: 'SBP'` means there's no column called `SBP`. Python is case-sensitive, and the column is `sbp`.
If Colab shows an **Explain error** button under the message, it asks Gemini to explain the error in plain language.

## 13. Read a cell the way an assistant writes it
Before you run this, answer four questions: Which rows go in? What's grouped? What's computed?
How many lines of output will there be?

In [ ]:
# Mean SBP by arm, among patients with SBP of 145 or more, highest first
(clinic[clinic["sbp"] >= 145]
    .groupby("arm")["sbp"]
    .mean()
    .sort_values(ascending=False)
    .round(1))

## Five questions for any cell, yours or the assistant's
1. **What goes in?** Which table, which columns?
2. **Which rows are kept or dropped?** Look for filters, `dropna()`, and `drop_duplicates()`.
3. **Does the row count change, and should it?** Look for `merge`, `groupby`, and `pivot`.
4. **What's computed?** A count, a mean, a model, and on which rows?
5. **Does the output match your prediction?** If not, find out why before you trust it.

You'll use all five in both labs today.